# Probability

In [1]:
import numpy as np

## 1. Probability Basics

**Probability** is the mathematical measure of how likely an event is to occur. It is always quantified as a number between 0 and 1 (or 0% and 100%):

- **0** means the event is impossible.
- **1** means the event is **certain** to happen.

**Core Terminology**
- **Experiment**: An action or process that leads to an uncertain outcome (e.g., flipping a coin, rolling a die).
- **Sample Space (S)**: The set of all possible outcomes. For a standard 6-sided die, \(S = \{1, 2, 3, 4, 5, 6\}\).
- **Event (E)**: A specific outcome or a set of outcomes we care about. For example, rolling an even number: \(E = \{2, 4, 6\}\).
- **Probability Formula**: For equally likely outcomes, the probability of an event P(E) is calculated as:

    $P(A) = \frac{\text{Favorable Outcomes}}{\text{Total Outcomes}}$

Using our die example, P(Event) = 3/6 = 0.5 (or 50%)

**NumPy Simulation: Law of Large Numbers**

In the real world, if you flip a coin 10 times, you might get 7 heads and 3 tails (70% heads). However, according to the **Law of Large Numbers**, as you repeat an experiment more times, the experimental probability will get closer and closer to the true theoretical probability (50%).


In [50]:
# Set a random seed for reproducibility
np.random.seed(42)

# 1. Define paramters
num_rolls = 1000000 # Simulate 1 million die rolls
sides_of_die = 6

# 2. Simulate the die rolls
# np.random.randint(low, high, size) picks integers from 'low' (inclusive) to 'high' (exclusive)
rolls = np.random.randint(1, sides_of_die + 1, size=num_rolls)

# 3. Calculate the experimental probability of rolling a '4'
# Count how many times the outcome was exactly 4
four_count = np.sum(rolls == 4)
experimental_prob = four_count / num_rolls

# 4. Theoretical probability
theoretical_prob = 1 / 6

# Print results:
print(f"Total Rolls: {num_rolls}")
print(f"Number of times '4' was rolled: {four_count:,}")
print(f"Experimental Probability: {experimental_prob:.6f}")
print(f"Theoretical Probability: {theoretical_prob:.6f}")
print(f"Difference: {abs(theoretical_prob - experimental_prob):.6f}")


Total Rolls: 1000000
Number of times '4' was rolled: 166,662
Experimental Probability: 0.166662
Theoretical Probability: 0.166667
Difference: 0.000005


## 2. Conditional Probability

**Conditional probability** is the probability of an event occurring **given that another event has already occurred**. It allows us to update our beliefs about the likelihood of an outcome based on new information or constraints. 

We write this mathematically as **$P(A \mid B)$**, which is read as *"the probability of event $A$ given event $B$."*

#### The Fundamental Formula
The formula for conditional probability is:

$$P(A \mid B) = \frac{P(A \cap B)}{P(B)}$$

Where:
* **$P(A \mid B)$** is the conditional probability of $A$ given $B$.
* **$P(A \cap B)$** is the **joint probability** (the probability that *both* events $A$ and $B$ happen together).
* **$P(B)$** is the probability of the condition event $B$ happening ($P(B) > 0$).

#### Visualizing the Concept: Shrinking the Sample Space
Imagine you have a full deck of cards. The total sample space is **52 cards**. 

* If you want to find the probability of drawing a **King** ($A$), it is $\frac{4}{52}$.
* Now, you receive a hint: the card drawn is a **Face Card** ($B$). 

Event $B$ has already happened, so your sample space instantly shrinks from 52 cards down to just the **12 face cards** (Jacks, Queens, Kings). Out of those 12 face cards, 4 are Kings. 

Therefore:
$$P(\text{King} \mid \text{Face Card}) = \frac{4}{12} = \frac{1}{3}$$


In [5]:
# Set random seed for reproducibility
np.random.seed(42)

# Total population size
population_size = 100_000

# 1. Generate baseline disease data (Assume 5% of the population actually has the disease)
# 1 = Has Disease, 0 = Healthy
has_disease = np.random.choice([0, 1], size=population_size, p=[0.95, 0.05])

# 2. Simulate diagnostic test results
# Initialize an empty array for test results (0 = Negative, 1 = Positive)
test_positive = np.zeros(population_size, dtype=int)

# Condition A: If they have the disease, the test is 90% accurate (True Positive Rate)
disease_indices = np.where(has_disease == 1)[0]
test_positive[disease_indices] = np.random.choice([0, 1], size=len(disease_indices), p=[0.10, 0.90])

# Condition B: If they are healthy, the test has a 10% false positive rate
healthy_indices = np.where(has_disease == 0)[0]
test_positive[healthy_indices] = np.random.choice([0, 1], size=len(healthy_indices), p=[0.90, 0.10])

# 3. Calculate Joint and Conditional Probabilities using the simulation data
# P(B): Total number of people who tested positive
total_positives = np.sum(test_positive == 1)

# P(A ∩ B): People who have the disease AND tested positive
true_positives = np.sum((has_disease == 1) & (test_positive == 1))

# P(A | B): Probability of having the disease GIVEN a positive test
conditional_prob = true_positives / total_positives

# Print Results
print(f"Total Population: {population_size:,}")
print(f"Total Positive Tests P(B): {total_positives:,}")
print(f"True Positives P(A ∩ B): {true_positives:,}")
print(f"P(Disease | Positive Test): {conditional_prob:.4f} ({conditional_prob * 100:.2f}%)")



Total Population: 100,000
Total Positive Tests P(B): 13,924
True Positives P(A ∩ B): 4,358
P(Disease | Positive Test): 0.3130 (31.30%)


## 3. Bayes' Theorem

**Bayes' Theorem** is a mathematical formula used to determine the conditional probability of an event based on prior knowledge and new evidence.

It adjusts probabilities when new information comes in and helps make better decisions in uncertain situations.

**The Formula**

$$P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)}$$

**Component Breakdown**
* **$P(A|B)$ (Posterior):** The probability of event $A$ occurring given that event $B$ is true.
* **$P(B|A)$ (Likelihood):** The probability of event $B$ occurring given that event $A$ is true.
* **$P(A)$ (Prior):** The initial probability of event $A$ before seeing the new evidence.
* **$P(B)$ (Marginal Likelihood):** The total probability of event $B$ occurring across all possible scenarios.


**Example**

Imagine a clinic tests patients for a rare disease that affects **1%** of the population. 

**The Parameters**

* **Sensitivity (True Positive Rate):** The test catches the disease **99%** of the time.
* **False Positive Rate:** **5%** of healthy people will accidentally test positive anyway.

If a patient **tests positive**, what is the actual probability that they have the disease?

---

**1. Define the Variables**

* **$P(D)$** (Prior probability of having the disease) = **0.01** (1%)
* **$P(H)$** (Prior probability of being healthy) = **0.99** (99%)
* **$P(+|D)$** (Likelihood of testing positive if sick) = **0.99** (99%)
* **$P(+|H)$** (Likelihood of testing positive if healthy) = **0.05** (5%)

---

**2. Calculate the Total Probability of Testing Positive: $P(+)$**

We must find the total percentage of the population that will test positive. This includes truly sick people who test positive *plus* healthy people who get a false positive.

$$P(+) = [P(+|D) \cdot P(D)] + [P(+|H) \cdot P(H)]$$

$$P(+) = [0.99 \cdot 0.01] + [0.05 \cdot 0.99]$$

$$P(+) = 0.0099 + 0.0495 = 0.0594$$

*(About 5.94% of the entire population will receive a positive test result.)*

---

**3. Apply Bayes' Theorem: $P(D|+)$**

Now, we calculate the probability of actually having the disease given that the test came back positive.

$$P(D|+) = \frac{P(+|D) \cdot P(D)}{P(+)}$$

$$P(D|+) = \frac{0.99 \cdot 0.01}{0.0594}$$

$$P(D|+) = \frac{0.0099}{0.0594} \approx 0.1667$$

---

**Final Verdict**

Despite the test being 99% accurate, there is only a **16.67% chance** that a person who tests positive actually has the disease. 

**Why is it so low?**

Because the disease is highly rare ($1\%$), the sheer volume of false positives from the massive healthy population ($5\%$ of the remaining $99\%$) completely outnumbers the true positive results.

In [51]:
def calculate_bayes(p_prior, p_likelihood_true, p_likelihood_false):
    """
    Calculates the posterior probability using Bayes' Theorem.
    
    Parameters:
    p_prior (float): Prior probability of the event occurring, P(A).
    p_likelihood_true (float): Probability of positive evidence given event occurred, P(B|A).
    p_likelihood_false (float): Probability of positive evidence given event did NOT occur, P(B|not A).
    
    Returns:
    float: Posterior probability, P(A|B).
    """
    # 1. Calculate prior probability of the negative case: P(not A)
    p_prior_neg = 1.0 - p_prior
    
    # 2. Vectorize inputs into NumPy arrays for structural clarity
    priors = np.array([p_prior, p_prior_neg])
    likelihoods = np.array([p_likelihood_true, p_likelihood_false])
    
    # 3. Calculate total marginal probability of the evidence: P(B)
    # This multiplies the arrays element-wise and sums them up
    p_marginal = np.sum(priors * likelihoods)
    
    # 4. Apply Bayes' Theorem formula: P(A|B) = [P(B|A) * P(A)] / P(B)
    p_posterior = (p_likelihood_true * p_prior) / p_marginal
    
    return p_posterior

# --- Medical Screening Sample Execution ---
# P(Disease) = 1%
prior_disease = 0.01 

# P(+ | Disease) = 99% (True Positive Rate)
test_sensitivity = 0.99 

# P(+ | Healthy) = 5% (False Positive Rate)
false_positive_rate = 0.05 

# Compute the probability of having the disease given a positive test result
probability_sick = calculate_bayes(
    p_prior=prior_disease, 
    p_likelihood_true=test_sensitivity, 
    p_likelihood_false=false_positive_rate
)

print(f"Total probability of testing positive: {np.sum(np.array([prior_disease, 1-prior_disease]) * np.array([test_sensitivity, false_positive_rate])):.4f}")
print(f"Probability of actually having the disease: {probability_sick:.4f} ({probability_sick * 100:.2f}%)")

Total probability of testing positive: 0.0594
Probability of actually having the disease: 0.1667 (16.67%)


## 4. Random Variable

In probability and machine learning, a random variable (usually denoted as a capital letter like $X$ or $Y$) is not actually a "variable" in the traditional algebraic sense. Instead, it is a **mathematical function that maps the outcomes of a random event to numerical values**.

For example, if you flip a coin twice, the raw outcomes are text combinations: `HH`, `HT`, `TH`, `TT`. A random variable $X$ can be defined as "the total number of Heads", which converts those text outcomes into numbers: `{2, 1, 1, 0}`.

### Types of Random Variables

#### 1. Discrete Random Variables
* Take on a **countable, distinct set of values** (usually whole numbers).
* *Examples*: Number of customer service calls in a day, number of heads in 10 coin flips, or whether an email is spam (0 or 1).
* **Probability Rule**: Described by a **Probability Mass Function (PMF)**, which gives the exact probability for each individual discrete value.

#### 2. Continuous Random Variables
* Take on an **infinite number of possible values** within a given continuous range or interval.
* **Examples**: Exact height of a person, temperature, stock prices, or model prediction errors.
* **Probability** Rule: Described by a **Probability Density Function (PDF)**. For continuous variables, the probability of getting one exact single value is technically zero; instead, we measure the probability of falling within a range (an integral under the curve).

### Random Variables in Machine Learning
* **Features as Random Variables**: Every column in a dataset (e.g., age, income, pixel brightness) is treated as a random variable sampled from an underlying probability distribution.
* **Targets/Labels as Random Variables**: The output we want to predict (e.g., house price or classification category) is also modeled as a random variable.

In [2]:
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

# --- 1. Discrete Random Variable Example ---
# Simulating the number of heads in 4 coin flips, repeated 1,000 times
# Here, the random variable X takes values from {0, 1, 2, 3, 4}
discrete_rv = np.random.binomial(n=4, p=0.5, size=1000)

print("Unique values observed for Discrete RV (Heads in 4 flips):", np.unique(discrete_rv))
print("Mean (Expected Value) of discrete RV:", np.mean(discrete_rv))


# --- 2. Continuous Random Variable Example ---
# Simulating continuous measurements, such as daily temperatures (Mean = 25°C, Std = 3°C)
continuous_rv = np.random.normal(loc=25.0, scale=3.0, size=1000)

print("\nFirst 5 values of Continuous RV (Temperatures):", continuous_rv[:5])
print("Mean of continuous RV:", np.mean(continuous_rv))
print("Standard Deviation of continuous RV:", np.std(continuous_rv))

Unique values observed for Discrete RV (Heads in 4 flips): [0 1 2 3 4]
Mean (Expected Value) of discrete RV: 1.971

First 5 values of Continuous RV (Temperatures): [25.533103   20.99396692 26.14059355 26.83175724 26.67937134]
Mean of continuous RV: 25.29668745384548
Standard Deviation of continuous RV: 2.965315324215403


## 5. Probability Distribution

A **probability distribution** is a mathematical function that describes how probabilities are distributed across all the possible values that a random variable can take.

* **For Discrete Variables**: It assigns a precise probability to each individual outcome using a **Probability Mass Function (PMF)**.
* **For Continuous Variables**: It describes the likelihood of falling within a range of values using a **Probability Density Function (PDF)**, where the total area under the curve equals $1$.

#### Why Probability Distributions Matter in Machine Learning
* **Data Modeling**: Algorithms often make assumptions about the underlying distribution of data (e.g., Linear Regression assumes error residuals are normally distributed).
* **Loss Functions**: Maximum Likelihood Estimation (MLE) uses distributions to define objective functions (like cross-entropy loss for classification, which stems from Bernoulli distributions).
* **Generative AI**: Modern models (like Diffusion Models or Variational Autoencoders) learn complex probability distributions to generate entirely new images, text, or audio.

#### Common Distributions Overview
1. Bernoulli Distribution: Models a single trial with two outcomes (success/failure, 0/1).
2. Binomial Distribution: Models the total number of successes in multiple independent Bernoulli trials.
3. Normal (Gaussian) Distribution: The famous bell-shaped curve characterized by a mean ($\mu$) and variance ($\sigma^2$), heavily relied upon due to the Central Limit Theorem.
4. Uniform Distribution: Every outcome within a given range has an equal probability of occurring.

In [3]:
import numpy as np

# Set seed for reproducibility
np.random.seed(42)

# --- 1. Uniform Distribution (Equal probability across a range) ---
# Simulate 1,000 rolls of a continuous uniform range between 0 and 10
uniform_samples = np.random.uniform(low=0.0, high=10.0, size=1000)
print("Uniform Distribution - Mean:", np.mean(uniform_samples), "(Expected: ~5.0)")


# --- 2. Bernoulli Distribution (Single trial binary outcome) ---
# Simulate a biased coin flip where probability of success (1) is 0.7, 1,000 times
bernoulli_samples = np.random.binomial(n=1, p=0.7, size=1000)
success_rate = np.mean(bernoulli_samples)
print(f"Bernoulli Distribution - Empirical Success Rate: {success_rate:.2f} (Expected: 0.70)")


# --- 3. Normal Distribution (Bell curve) ---
# Simulate features like heights with mean=170cm and std=5cm
normal_samples = np.random.normal(loc=170.0, scale=5.0, size=1000)
print(f"Normal Distribution - Mean: {np.mean(normal_samples):.2f}, Std: {np.std(normal_samples):.2f}")

Uniform Distribution - Mean: 4.902565533201336 (Expected: ~5.0)
Bernoulli Distribution - Empirical Success Rate: 0.69 (Expected: 0.70)
Normal Distribution - Mean: 170.06, Std: 4.89


## 6. Normal Distribution

The **Normal Distribution** (also known as the **Gaussian distribution** or the **bell curve**) is one of the most important probability distribution in statistics and machine learning.

* **The Shape:** It is a continuous, symmetrical, bell-shaped curve where the highest point (the peak) represnets the most likely outcome (the mean). The probabilities taper off symmetrically as you move away from the center in either direction.
* **Symmetry:** The mean, median, and mode all fall at the exact same central point.

### The Two Defining Parameters
A normal distribution is completely defined by just two parameters:
1. **Mean ($\mu$)**: Determines the **location** or center of the peak on the axis.
2. **Variance ($\sigma^2$) or Standard Deviation ($\sigma$)**: Determines the **spread** or width of the bell curve. A small standard deviation creates a tall, narrow peak (data is tightly clustered), while a large standard deviation creates a flat, wide curve (data is widely dispersed).

### The Empirical Rule (The 68-95-99.7 Rule)
For any normal distribution, data follows a very predictable spread relative to the standard deviation ($\sigma$) from the mean ($\mu$):
* **~68%** of the data falls within $1$ **standard deviation** ($\mu \pm 1\sigma$).
* **~95%** of the data falls within $2$ **standard deviations** ($\mu \pm 2\sigma$).
* **~99.7%** of the data falls within $3$ **standard deviations** ($\mu \pm 3\sigma$).

### Why the Normal Distribution Matters in Machine Learning
* **Central Limit Theorem (CLT)**: This theorem states that when you add together a large number of independent random variables, their sum tends to follow a normal distribution—regardless of the distribution of the individual variables. This is why natural phenomena (heights, errors, test scores) often look normal.
* **Error Modeling**: Linear regression and many other algorithms assume that prediction errors (residuals) are normally distributed.
* **Gaussian Naive Bayes**: A classification algorithm that assumes the features within each class follow a normal distribution.
* **Weight Initialization**: Neural network weights are often initialized by drawing values from a normal distribution to break symmetry during training.

In [2]:
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

# Define parameters: Mean height = 170 cm, Standard Deviation = 5 cm
mu = 170.0
sigma = 5.0

# Generate 10,000 samples from this normal distribution
heights = np.random.normal(loc=mu, scale=sigma, size=10000)

# 1. Check empirical mean and standard deviation
empirical_mean = np.mean(heights)
empirical_std = np.std(heights)

print(f"Target Mean: {mu} | Empirical Mean: {empirical_mean:.2f}")
print(f"Target Std: {sigma} | Empirical Std: {empirical_std:.2f}")

# 2. Verify the Empirical Rule (68-95-99.7 rule)
# Count how many samples fall within 1, 2, and 3 standard deviations
within_1_sigma = np.sum((heights >= mu - sigma) & (heights <= mu + sigma)) / len(heights)
within_2_sigma = np.sum((heights >= mu - 2*sigma) & (heights <= mu + 2*sigma)) / len(heights)
within_3_sigma = np.sum((heights >= mu - 3*sigma) & (heights <= mu + 3*sigma)) / len(heights)

print(f"\nPercentage within 1 std dev: {within_1_sigma * 100:.1f}% (Expected: ~68%)")
print(f"Percentage within 2 std dev: {within_2_sigma * 100:.1f}% (Expected: ~95%)")
print(f"Percentage within 3 std dev: {within_3_sigma * 100:.1f}% (Expected: ~99.7%)")

Target Mean: 170.0 | Empirical Mean: 169.99
Target Std: 5.0 | Empirical Std: 5.02

Percentage within 1 std dev: 68.2% (Expected: ~68%)
Percentage within 2 std dev: 95.3% (Expected: ~95%)
Percentage within 3 std dev: 99.7% (Expected: ~99.7%)


## 7. Bernoulli distribution

The **Bernoulli distribution** is the simplest discrete probability distribution. It models a **single experiment (trial) that has only two possible outcomes**: success (usually coded as $1$) or failure (usually coded as $0$).
* **The Single Trial**: Unlike a coin that you might flip 10 times, a Bernoulli trial happens once.
* **Examples**: Flipping a coin once (Heads = 1, Tails = 0), an email being spam (Yes = 1, No = 0), or a customer making a purchase during a visit (Yes = 1, No = 0).

### The Defining Parameter ($p$)
A Bernoulli distribution is controlled by a single parameter:
* $p$: The probability of success (where $0 \le p \le 1$).
* Consequently, the probability of failure is $1 - p$ (often denoted as $q$).

### Key Mathematical Properties
* Expected Value (Mean): $E[X] = p$
* Variance: $\text{Var}(X) = p(1 - p)$

### Why the Bernoulli Distribution Matters in Machine Learning
* **Binary Classification:** Many machine learning tasks involve predicting a binary outcome (e.g., churn vs. stay, click vs. no click).
* **Logistic Regression:** The final output of a logistic regression model is a probability score between $0$ and $1$, representing the parameter $p$ of an underlying Bernoulli distribution.
* **Loss Functions: Binary Cross-Entropy Loss—**the standard loss function for binary classification neural networks—is mathematically derived from the Bernoulli likelihood function.
* **Building Block:** If you take multiple independent Bernoulli trials and count the total number of successes, you get a Binomial distribution.

In [3]:
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

# Define the probability of success (e.g., 70% chance a user clicks an ad)
p = 0.7

# 1. Simulate a single Bernoulli trial
single_trial = np.random.binomial(n=1, p=p)
print("Single Trial Outcome (1 = Success, 0 = Failure):", single_trial)

# 2. Simulate 10,000 Bernoulli trials (e.g., 10,000 ad impressions)
trials = np.random.binomial(n=1, p=p, size=10000)

# Calculate empirical mean and variance
empirical_mean = np.mean(trials)
empirical_variance = np.var(trials)

print(f"\nTarget Probability (p): {p}")
print(f"Empirical Mean (Observed p): {empirical_mean:.4f} (Expected: {p})")
print(f"Empirical Variance: {empirical_variance:.4f} (Expected: {p * (1 - p):.4f})")

Single Trial Outcome (1 = Success, 0 = Failure): 1

Target Probability (p): 0.7
Empirical Mean (Observed p): 0.7113 (Expected: 0.7)
Empirical Variance: 0.2054 (Expected: 0.2100)


## 8. Binomial distribution

If a **Bernoulli distribution** models a single trial (like flipping a coin once), the **Binomial distribution** models the **total number of successes across multiple independent trials.**

* **The Setup:** Imagine you repeat a Bernoulli trial $n$ times (e.g., flipping a coin 10 times, or sending 100 marketing emails). Each trial is independent, and the probability of success ($p$) remains constant for every trial.
* **The Random Variable ($X$):** Represents the total count of successes (e.g., getting exactly 6 heads out of 10 flips, or 12 opens out of 100 emails).

### The Two Defining Parameters
A binomial distribution is controlled by two parameters:
* $n$: The total number of independent trials.
* $p$: The probability of success on any given single trial.

### Key Mathematical Properties
* **Expected Value (Mean):** $E[X] = n \cdot p$ (If you flip a coin 10 times where $p=0.5$, you expect on average $5$ heads).
* **Variance:** $\text{Var}(X) = n \cdot p \cdot (1 - p)$

### Why the Binomial Distribution Matters in Machine Learning
* **A/B Testing & Hypothesis Testing:** Determining whether a new website design or machine learning model produces a statistically significant higher conversion rate than the old one.
* **Accuracy Metrics:** Evaluating the performance of a classifier over a test set can be viewed through binomial trials (each prediction is either correct or incorrect).
* **Ensemble Methods:** Models like Random Forests use majority voting across multiple trees, which can be modeled using binomial probabilities.

In [4]:
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

# Scenario: A multiple-choice test has 20 questions. 
# Each question has 4 options, so guessing randomly gives a probability of success p = 0.25.
n_questions = 20
p_success = 0.25

# Simulate a student guessing randomly across all 20 questions, repeated 10,000 times
simulated_scores = np.random.binomial(n=n_questions, p=p_success, size=10000)

# Calculate empirical mean and variance
empirical_mean = np.mean(simulated_scores)
empirical_variance = np.var(simulated_scores)

print(f"Target Expected Value (n * p): {n_questions * p_success}")
print(f"Empirical Mean Score: {empirical_mean:.2f}\n")

print(f"Target Variance (n * p * (1-p)): {n_questions * p_success * (1 - p_success):.2f}")
print(f"Empirical Variance: {empirical_variance:.2f}")

# Find the percentage of times a random guesser scored 7 or more correct answers
passing_score_count = np.sum(simulated_scores >= 7)
passing_probability = passing_score_count / len(simulated_scores)
print(f"\nProbability of getting 7+ correct by random guessing: {passing_probability * 100:.2f}%")

Target Expected Value (n * p): 5.0
Empirical Mean Score: 4.96

Target Variance (n * p * (1-p)): 3.75
Empirical Variance: 3.68

Probability of getting 7+ correct by random guessing: 20.82%


## 9. Expected Value

The **Expected Value** (denoted as $E[X]$ or $\mu$) is the theoretical long-term average or mean of a random variable. If you repeat a random experiment thousands or millions of times, the average of all outcomes will converge toward the expected value.

* **Not a Single Guaranteed Outcome:** The expected value does not have to be one of the actual possible values you can observe. For example, the expected number of children per household in a region might be $1.9$, even though no individual family has $1.9$ children.
* **Weighted Average:** It is calculated by taking every possible outcome, multiplying it by its respective probability, and summing them all up.

### Mathematical Formulation
* For Discrete Random Variables:
$$E[X] = \sum_{x} x \cdot P(X = x)$$
*(Sum of each outcome multiplied by its probability mass function).*
* For Continuous Random Variables:
$$E[X] = \int_{-\infty}^{\infty} x \cdot f(x) \, dx$$
(Integral of each value multiplied by its probability density function).

### Why Expected Value Matters in Machine Learning
* **Loss Functions:** Training a machine learning model involves minimizing an objective or loss function. Many loss functions (like Mean Squared Error) are designed to estimate the expected value of the target variable given the inputs $E[Y \mid X]$.
* **Reinforcement Learning:** Agents make decisions based on the expected future reward of taking a specific action in a given state.
* **Decision Making:** Expected values help algorithms choose the optimal action or class boundary under uncertainty.

In [5]:
import numpy as np

# Scenario: A carnival game where you draw a card from a deck or roll a custom die.
# Let's define the payouts (outcomes, X) and their respective probabilities P(X):
# - Roll 1: Lose $5 (Probability: 0.1)
# - Roll 2: Win $2  (Probability: 0.4)
# - Roll 3: Win $10 (Probability: 0.5)

outcomes = np.array([-5.0, 2.0, 10.0])
probabilities = np.array([0.1, 0.4, 0.5])

# Expected Value is the sum of (outcomes * probabilities)
# In NumPy, we can compute this efficiently using the dot product or element-wise multiplication + sum
expected_value = np.dot(outcomes, probabilities)

print(f"Outcomes: {outcomes}")
print(f"Probabilities: {probabilities}")
print(f"Calculated Expected Value (E[X]): ${expected_value:.2f}")

# Simulating the game 10,000 times to show convergence to the Expected Value
np.random.seed(42)
simulated_draws = np.random.choice(outcomes, size=10000, p=probabilities)
empirical_average = np.mean(simulated_draws)

print(f"\nEmpirical Average over 10,000 simulations: ${empirical_average:.2f}")

Outcomes: [-5.  2. 10.]
Probabilities: [0.1 0.4 0.5]
Calculated Expected Value (E[X]): $5.30

Empirical Average over 10,000 simulations: $5.22


## 10. Variance

While the E**xpected Value** tells you where the center or average of a random variable lies, **Variance** (denoted as $\text{Var}(X)$ or $\sigma^2$) tells you **how spread out or dispersed the values are** around that center.
* **High Variance:** The data points are widely scattered across a broad range of values. Predictions or measurements are unpredictable and erratic.
* **Low Variance:** The data points are tightly clustered very close to the expected value. Measurements are consistent and reliable.

### The Mechanics of Variance
To calculate how far values are from the mean ($\mu = E[X]$), you might think to just average the differences $(x - \mu)$. However, values below the mean will have negative differences and values above will have positive ones, causing them to cancel each other out to zero.

To fix this, variance squares each difference:
1. Find the difference between each outcome and the mean: $(x - \mu)$
2. **Square the difference:** $(x - \mu)^2$ (This makes all values positive and heavily penalizes large deviations).
3. Take the expected value (weighted average) of those squared differences.

### Mathematical Formulation
* Discrete Random Variable:
$$\text{Var}(X) = E[(X - \mu)^2] = \sum (x - \mu)^2 \cdot P(X = x)$$
* **Standard Deviation ($\sigma$):** Because variance is squared, its units are also squared (e.g., dollars squared). To get back to the original units, we take the square root of the variance, known as the **Standard Deviation** ($\sigma = \sqrt{\text{Var}(X)}$).

### Why Variance Matters in Machine Learning
* **Overfitting vs. Underfitting:** High-variance models (like overly complex decision trees or deep neural networks with no regularization) memorize the training data, including noise, failing to generalize to new data.
* **Regularization:** Techniques like L1 and L2 regularization directly penalize the variance of model weights to keep them stable.
* **Feature Scaling:** Algorithms like Gradient Descent converge much faster when features have similar variance scales.

In [6]:
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

# Scenario: Exam scores for a class of 5 students
scores = np.array([70, 75, 80, 85, 90])

# 1. Calculate Expected Value (Mean)
mean_score = np.mean(scores)
print(f"Mean Score (Expected Value): {mean_score}")

# 2. Calculate Variance manually: E[(X - mean)^2]
squared_differences = (scores - mean_score) ** 2
manual_variance = np.mean(squared_differences)
print(f"Manual Variance: {manual_variance:.2f}")

# 3. Calculate Variance using NumPy's built-in function
# Note: NumPy defaults to population variance (ddof=0). 
# For sample variance, use ddof=1 (Degrees of Freedom).
np_population_variance = np.var(scores)
np_sample_variance = np.var(scores, ddof=1)

print(f"NumPy Population Variance (ddof=0): {np_population_variance:.2f}")
print(f"NumPy Sample Variance (ddof=1): {np_sample_variance:.2f}")

# 4. Standard Deviation is just the square root of variance
std_dev = np.std(scores, ddof=1)
print(f"Sample Standard Deviation: {std_dev:.2f}")

Mean Score (Expected Value): 80.0
Manual Variance: 50.00
NumPy Population Variance (ddof=0): 50.00
NumPy Sample Variance (ddof=1): 62.50
Sample Standard Deviation: 7.91


### 11. Covariance

While variance measures how a **single** random variable spreads out, **covariance** measure **how two different random variables change together**. It tells you whether they share a linear relationship.
* **Positive Covariance ($> 0$):** When one variable increases, the other variable tends to also increase (they move in the same direction). For example, hours spent studying and exam scores.
* **Negative Covariance ($< 0$):** When one variable increases, the other tends to decrease (they move in opposite directions). For example, hours spent watching TV and exam scores.
* **Zero Covariance ($\approx 0$):** There is no linear relationship between the two variables; they vary independently.

### Mathematical Formulation
The covariance between two random variables $X$ and $Y$ is defined as the expected value of the product of their deviations from their respective means ($\mu_X$ and $\mu_Y$):
$$\text{Cov}(X, Y) = E[(X - \mu_X)(Y - \mu_Y)]$$
* **Limitation**: The raw value of covariance is heavily influenced by the scale of the units (e.g., measuring income in dollars versus thousands of dollars changes the covariance magnitude). To fix this, we normalize covariance by dividing it by the product of the standard deviations, giving us **Correlation** (Pearson's $r$, bounded between $-1$ and $1$).

### Why Covariance Matters in Machine Learning
* **Feature Analysis:** Covariance helps data scientists understand feature collinearity (when two features carry redundant information).
* **Principal Component Analysis (PCA):** A foundational dimensionality reduction technique that relies entirely on computing the covariance matrix of features to find directions of maximum variance.
* **Multivariate Normal Distribution:** Covariance parameters define the spread and orientation of multidimensional bell curves.

In [7]:
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

# Scenario: Compare Hours Studied (X) vs. Exam Scores (Y)
hours_studied = np.array([1, 2, 3, 4, 5])
exam_scores = np.array([50, 55, 65, 70, 85])

# 1. Calculate Covariance using np.cov()
# Note: np.cov returns a 2x2 Covariance Matrix because it calculates 
# the covariance of every variable against every other variable.
# Matrix structure:
# [[ Cov(X,X)  Cov(X,Y) ],
#  [ Cov(Y,X)  Cov(Y,Y) ]]  <- Note: Cov(X,X) is just the variance of X!

cov_matrix = np.cov(hours_studied, exam_scores)

print("Covariance Matrix:\n", cov_matrix)

# Extract the specific covariance between X and Y (row 0, column 1)
covariance_xy = cov_matrix[0, 1]
print(f"\nCovariance between Hours Studied and Exam Scores: {covariance_xy:.2f}")

# 2. Compare with a negatively correlated variable (e.g., Hours of Sleep Lost vs. Exam Scores)
sleep_lost = np.array([5, 4, 3, 2, 1])
neg_cov_matrix = np.cov(sleep_lost, exam_scores)
print(f"Covariance between Sleep Lost and Exam Scores: {neg_cov_matrix[0, 1]:.2f} (Negative relationship)")

Covariance Matrix:
 [[  2.5   21.25]
 [ 21.25 187.5 ]]

Covariance between Hours Studied and Exam Scores: 21.25
Covariance between Sleep Lost and Exam Scores: -21.25 (Negative relationship)
